# 🎲 AI Gamemaster - OpenEnv RL Training with Unsloth

This notebook trains a Large Language Model to act as a **Rules-Compliant Gamemaster** using **GRPO (Group Relative Policy Optimization)** and **Unsloth** for 2x faster training.

### Theme Alignment: Self-Improvement & World Modeling
The agent starts with zero knowledge of how to format strict JSON state updates. Through interaction with the `GamemasterEnv`, it receives rewards for correctly interpreting dice rolls, applying damage, and managing inventory. Over time, the model **self-improves**, learning to seamlessly blend creative narrative with rigorous, programmatic rule enforcement.

In [ ]:
%%capture
# 1. Install Unsloth and dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

# 2. Install OpenEnv to connect to our Environment
!pip install openenv-core pydantic

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

from trl import GRPOTrainer, GRPOConfig
import asyncio
import re
import json

max_seq_length = 2048
lora_rank = 16
gpu_memory_utilization = 0.7

# We use Qwen2.5-1.5B as a fast, capable base model for the hackathon
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True, # 4-bit quantization for memory efficiency on Colab T4
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=gpu_memory_utilization,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth", 
    random_state=3407,
)

### Define the OpenEnv Reward Function

This is the bridge between Hugging Face TRL and OpenEnv. 
For every generation the model produces, we parse the JSON action, send it to the `GamemasterEnv` server, and retrieve the reward.

In [ ]:
# IMPORTANT: You must have your OpenEnv running! 
# If running locally: `uvicorn server.app:app --port 8000`
# If deployed to HF Spaces, replace with your Space URL.
ENV_URL = "http://localhost:8000"  # Or "https://<your-username>-gamemaster-env.hf.space"

# Import the client and action from your local environment package
# (Assuming you have cloned your env repo into the colab environment)
import sys
import os
# If you cloned your env into the colab workspace:
# sys.path.append(os.path.abspath('./gamemaster_env'))
from client import GamemasterEnv
from models import GamemasterAction

def openenv_reward_func(prompts, completions, **kwargs):
    rewards = []
    
    # TRL GRPO passes a batch of completions.
    for prompt, completion in zip(prompts, completions):
        try:
            # Extract the generated text
            generated_text = completion[0]["content"]
            
            # Parse the JSON from the LLM output
            # The LLM is prompted to output a JSON block containing the GamemasterAction fields
            json_match = re.search(r'\{.*\}', generated_text, re.DOTALL)
            if not json_match:
                rewards.append(-2.0) # Heavy penalty for not outputting JSON
                continue
                
            action_data = json.loads(json_match.group(0))
            action = GamemasterAction(**action_data)
            
            # Connect to the environment and take a step
            with GamemasterEnv(base_url=ENV_URL).sync() as env_client:
                # For simplicity in this stateless reward function, we reset before stepping.
                # For true Long-Horizon, you'd maintain the EnvClient session across steps in a custom rollout loop.
                env_client.reset()
                result = env_client.step(action)
                
                # The environment computes the reward based on rules (e.g. did it apply damage on a hit?)
                rewards.append(result.reward)
                
        except Exception as e:
            # Penalty for malformed actions, missing fields, or connection errors
            rewards.append(-1.0) 
            
    return rewards

### Create the Training Dataset
We seed the GRPO trainer with a variety of game states. The model must learn to read the `system_dice_roll` and `player_input` to decide its action.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """
You are an AI Gamemaster. You must enforce the rules of the game while telling a good story.
You will receive an Observation containing the player's action and the system's dice roll (1-20).
If the player attacks, a roll of >= 10 is a hit. You must apply damage.
If the roll is < 10, it is a miss. Do not apply damage.

You MUST respond ONLY with a valid JSON object matching this schema:
{
  "narrative_response": "Your story text here",
  "target_to_damage": "goblin" or null,
  "damage_amount": integer (0 if miss, >0 if hit),
  "item_to_give": "item name" or null
}
"""

# Create synthetic observation scenarios to train on
scenarios = [
    {"observation": "Player: 'I attack the goblin!' | Dice Roll: 15 | Goblin HP: 10"},
    {"observation": "Player: 'I swing my sword at the goblin.' | Dice Roll: 4 | Goblin HP: 10"},
    {"observation": "Player: 'I search the dead goblin for loot.' | Dice Roll: 12 | Goblin HP: 0"},
    {"observation": "Player: 'I attack the goblin!' | Dice Roll: 20 | Goblin HP: 10"},
]

dataset_dict = {"prompt": []}
for s in scenarios * 50: # Duplicate scenarios to create a dataset
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": s["observation"]}
    ]
    dataset_dict["prompt"].append(prompt)

train_dataset = Dataset.from_dict(dataset_dict)

In [ ]:
training_args = GRPOConfig(
    use_vllm = False, # Set True if you have a multi-GPU setup
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    logging_steps = 1,
    bf16 = True, # Use bf16 if on Ampere (A100) or newer, else fp16
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations = 4, # GRPO samples multiple completions to compute relative advantage
    max_prompt_length = 512,
    max_completion_length = 512,
    num_train_epochs = 1,
    save_steps = 50,
    max_grad_norm = 0.1,
    report_to = "none", # Change to 'wandb' for tracking
    output_dir = "outputs",
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        openenv_reward_func
    ],
    args = training_args,
    train_dataset = train_dataset,
)

trainer.train()

In [ ]:
# Save the trained LoRA adapters
model.save_pretrained("lora_gamemaster")
tokenizer.save_pretrained("lora_gamemaster")

print("Training complete! The Gamemaster has learned the rules of the dungeon.")